<a href="https://colab.research.google.com/github/mgd-code/llm-workshop-witten/blob/main/climate_classification_deepseek_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automating classification with open-weight models: DeepSeek via Fireworks AI

This notebook repeats the climate speech classification from the Anthropic notebook, but uses a DeepSeek model hosted by Fireworks AI instead. The prompt and data are identical, so the only things that change are the model and the code that calls it.

**Before running:**
1. Save a local copy of this .ipynb file in your Google drive so you can access it again later
2. Click the key icon in the left sidebar (**Secrets**), add a secret named `FIREWORKS_API_KEY` with your Fireworks API key as the value, and switch on **Notebook access**.
3. No GPU is needed for this notebook, because the model runs on Fireworks' servers, not in Colab. The default CPU runtime is fine (**Connect to Runtime**).

## 1. Basic setup: install and load packages


In [ ]:
# Install the Fireworks library
!pip install fireworks-ai

In [ ]:
# Import packages that are already installed

import pandas as pd # pandas package used for data analysis
from fireworks import Fireworks # fireworks package, used to call models hosted by Fireworks AI
from pathlib import Path # helps with calling different folder and file names
from google.colab import userdata, files # allows us to call the API key and download files

In [ ]:
# Import the API key from Colab Secrets
api_key = userdata.get("FIREWORKS_API_KEY")

## 2. Load the data from GitHub

The workshop files live in a GitHub repository. The cell below downloads them into this Colab session. It only downloads if the folder isn't already there, so it is safe to re-run.

In [ ]:
# Download the workshop files into this Colab session and set the project_root
!git clone https://github.com/mgd-code/llm-workshop-witten.git
project_root = Path("/content/llm-workshop-witten")

In [ ]:
# load climate dataset
climate_dataset_10 = pd.read_csv(project_root/"climate_sample_10.csv")

# check to make sure the file loaded
climate_dataset_10

## 3. Define the prompt

This is exactly the same prompt we used with Claude. Keeping the prompt identical lets us compare models fairly.

In [ ]:
instructions = """
    You are coding excerpts from central bank speeches. Read the excerpt and decide how it frames climate change (including "green transition" or "green finance").

Choose ONE category:

N. NOT ABOUT CLIMATE: the excerpt does not discuss climate change.
A. PASSING MENTION: climate change appears only as one item in a list of challenges or topics, and nothing specific is said about it.
B. FINANCIAL RISK: climate change as a risk to banks, insurers or the financial system (supervision, stress tests, disclosure, risk management).
C. MACROECONOMIC: climate change as a driver of economic outcomes (inflation, growth, prices, harvests, uncertainty for policymakers).
D. INVESTMENT & OPPORTUNITY: financing the transition, green finance markets, innovation, business opportunities.
E. COLLECTIVE ACTION: climate change as a global or societal problem that needs cooperation, government policy or moral responsibility (market failure, multilateralism, future generations, fairness between countries).

Rules:
- Choose A whenever climate change is only listed alongside other issues, even if the list is about risks or the economy.
- If more than one of B–E fits, choose the one the excerpt emphasizes most.

Do NOT explain your answer. Respond ONLY with the letter that corresponds with the chosen category. For example: "A"

This is the excerpt:


    """

## 4. Create the client

Just like with Anthropic, we create a client that sends our requests to the provider (here, Fireworks AI) using our API key, and passes the responses back to us.

Fireworks hosts many open-weight models, including DeepSeek. You can browse the available models and their official model IDs at https://fireworks.ai/models

In [ ]:
client = Fireworks(api_key=api_key)

## 5. Try automating a single example

First, we'll save a single example in an object called "excerpt"

In [ ]:
excerpt = """
There are a wealth of initiatives at present: the G20, the United Nations, the Financial Stability Board and the OECD are all involved. And notable within the EU, undoubtedly, is the European Commission's action plan on sustainable finance. In the case of Spain, we should mention the Draft Climate Change Law.

"""

Now we send a request to DeepSeek using "client.chat.completions.create", and save the result in an object called "response". The messages object works the same way as before: we send our instructions and the excerpt in the user role, and the model answers as the assistant.

A few settings to note:
- `model`: the official ID of the model on Fireworks.
- `reasoning_effort`: DeepSeek-V4.1-Flash is a reasoning model, which means it can "think" before answering. `"high"` turns thinking on.
- `max_tokens`: caps the length of the response. For reasoning models, the thinking counts toward this limit too, so it needs to be generous. It is a ceiling, not a target.
- `temperature`: 0 makes the output as consistent as possible, which is what we want when coding text.

In [ ]:
messages = [
    {
        "role": "user",
        "content": instructions + excerpt
    }
]

response = client.chat.completions.create(
    model="accounts/fireworks/models/deepseek-v4p1-flash-1017",
    messages=messages,
    reasoning_effort="high", # turn thinking on
    max_tokens=10000, # thinking + answer must fit within this limit
    temperature=0
)

The structure of the response differs a little from Anthropic's. Here, the answer is in `response.choices[0].message.content`: `[0]` selects the first (and only) answer, and `.content` pulls out its text.

In [ ]:
response.choices[0].message.content

Because DeepSeek is a reasoning model, we can also look at its thinking, which is kept separate from the answer. Even though our prompt says not to explain the answer, the model still reasons internally first. The token counts below include those thinking tokens, which is worth keeping in mind when managing costs.

In [ ]:
# The model's thinking (if it was returned)
print(getattr(response.choices[0].message, "reasoning_content", "No reasoning returned"))

# Token usage (output tokens include the thinking tokens)
print("Input tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)

## 6. Automate a bigger sample with a loop

The process here is exactly the same as in step 5. However, we put it in a loop instead, so that the client will run every one of the "text" column entries in our climate_dataset_10 file, and save the results to a csv file.

In [ ]:
# Initialize an empty list to store DeepSeek outputs, and a counter for token usage
deepseek_outputs = []
total_tokens = 0

# Iterate over each row in the dataframe
for index, row in climate_dataset_10.iterrows():
    # Extract the text from the dataframe
    text = row['text']

    messages = [
        {
            "role": "user",
            "content": instructions + text
        }
    ]

    # Classify, then store the result
    response = client.chat.completions.create(
        model="accounts/fireworks/models/deepseek-v4p1-flash-1017",
        messages=messages,
        reasoning_effort="high",
        max_tokens=10000,
        temperature=0
    )

    # Extract the response from DeepSeek
    answer = (response.choices[0].message.content or "").strip()

    # Append the classification to the list
    deepseek_outputs.append(answer)

    # Keep track of token usage
    total_tokens += response.usage.prompt_tokens + response.usage.completion_tokens


# Print the number of rows classified and the tokens used
print(f"Number of rows classified: {len(deepseek_outputs)}")
print(f"Total tokens used: {total_tokens}")

# Save outputs in a new column in our dataframe
climate_dataset_10['code_deepseek'] = deepseek_outputs

In [ ]:
climate_dataset_10

In [ ]:
# Export to csv and save locally
climate_dataset_10.to_csv(project_root/'deepseek_classification.csv', index=False)
files.download(str(project_root/'deepseek_classification.csv'))

## 7. Other options in client.chat.completions.create()

One option of particular interest is `reasoning_effort`. Setting it to `"none"` turns thinking off, so the model answers directly. This is usually faster and cheaper (far fewer tokens), but may be less accurate on tricky excerpts. Compare the output tokens below with the single example in step 5.

In [ ]:
messages = [
    {
        "role": "user",
        "content": instructions + excerpt
    }
]

response = client.chat.completions.create(
    model="accounts/fireworks/models/deepseek-v4p1-flash-1017",
    messages=messages,
    reasoning_effort="none", # turn thinking off
    max_tokens=10000,
    temperature=0
)

print("Answer:       ", response.choices[0].message.content)
print("Output tokens:", response.usage.completion_tokens)